# Instruction

The input data should be a table such as:
smiles     | solvent | property
-------- | ----- | -----
SMILES of Molecule  | SMILES of solvent | property value

Some solvents are not supported by GBRT and the input of schnet is a coordinate db instead of SMILES.


### System Check

### DataBase Searching

### Model Training

### Prediction and Evaluation

# System Check

In [ ]:
import torch
torch_version = torch.__version__
gpu = torch.cuda.is_available()
print(f'Troch version: {torch_version}, GPU status: {gpu}')

import FLAME
print(f'FLAME verion: {FLAME.__version__}')

print('Check Done!')

# Database Searching

In [ ]:
from FLAME import run_search

SMILES = 'COc1cc(OC)c2ccc(=O)oc2c1'
similarity_limit = 0.5
fingerprinter_type = 'morgan'

result_df = run_search(SMILES, similarity_limit, fingerprinter_type)
result_df

# Model Training

In [ ]:
# make schnet data
import pandas as pd
from FLAME.utils import get_schnet_data

data_path = 'data/FluoDB/abs_test.csv'
df = pd.read_csv(data_path)
df = df[df['smiles'].str.len() < 40]
df = df[df['smiles'].str.find('+') == -1]
df = df[df['smiles'].str.find('-') == -1]
df = df.drop_duplicates(subset=['smiles'])

get_schnet_data(df['smiles'].values, np.zeros(len(smiles)), df.index.values, save_path=data_path+'.db', conf_num=1)

In [ ]:
import os
from FLAME import flsf_train as training
# from FLAME import uvvisml_train as training
# from FLAME import abtmpnn_train as training
# from FLAME import fcnn_train as training
# from FLAME import gbrt_train as training
# from FLAME import schnet_train as training

epoch = 1
train_data = 'data/FluoDB/abs_train.csv'
valid_data = 'data/FluoDB/abs_valid.csv'
test_data = 'data/FluoDB/abs_test.csv.db'
model_save_path = 'model/test'

if os.path.exists(model_save_path):
    print(model_save_path , ' exists!')

training(model_save_path, train_data, valid_data, test_data, epoch)

# Prediction

In [ ]:
# Predict single
import pandas as pd
import numpy as np
from FLAME import flsf_predict as predict
# from FLAME import uvvisml_predict as predict
# from FLAME import abtmpnn_predict as predict
# from FLAME import fcnn_predict as predict
# from FLAME import gbrt_predict as predict
# from FLAME import schnet_predict as predict

solvent = ['O', 'CS(C)=O','C(Cl)Cl', 'CCO']
smiles = ['O=C1OC2=CC=C(C=C2C3=C1N=CO3)N','O=C1OC2=CC(N)=CC=C2C3=C1N=CO3']

df = pd.DataFrame({
    'smiles': sorted((smiles*len(solvent))),
    'solvent': solvent*len(smiles)
})
target = 'abs'
model_path = f'../model/test/fold_0/model_0'
output_file = 'test.csv'

df[f'{target}_pred'] = predict(model_path, output_file, smiles=df[['smiles','solvent']].values.tolist())
if target == 'e':
    df[f'{target}_pred'] = np.log10(df[f'{target}_pred'])

In [ ]:
# Predict File
import pandas as pd
from FLAME import flsf_predict as predict
# from FLAME import uvvisml_predict as predict
# from FLAME import abtmpnn_predict as predict
# from FLAME import fcnn_predict as predict
# from FLAME import gbrt_predict as predict
# from FLAME import schnet_predict as predict

save = False

target = 'abs'
model_path = f'model/flsfluoDB_{target}'
input_file = f'data/FluoDB/{target}_test.csv'
output_file = 'pred/test.csv'

df = pd.read_csv(input_file)
df['pred'] = predict(model_path, output_file,smiles=df[['smiles','solvent']].values.tolist())

# Evaluate

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

res = []
idx = []

solvent = -1
# solvent = 4

for target in ['abs', 'emi', 'plqy', 'e']:
    for m in ['GBRT', 'FCNN', 'uvvisml', 'schnet', 'abtmpnn' , 'flsf_maccs', 'flsf_morgan', 'flsf:
        idx.append((target, m))
        base = pd.read_csv(f'data/FluoDB/{target}_test.csv').iloc[:,-1]
        pred_df = pd.read_csv(f'pred/{m}/{m}_{target}.csv')
        if 'pred' not in pred_df.columns:
            pred_df['pred'] = pred_df[target]
            pred_df[target] = base
        if target == 'e':
            pred_df['pred'] = np.log10(pred_df['pred'])
            pred_df[target] = np.log10(pred_df[target])

        pred_df = pred_df.dropna()
        
        if solvent == 1: #single solvent
            pred_df = pred_df[pred_df['solvent']=='ClCCl'] 
        elif solvent > 1:
            for smi, sdf in pred_df.groupby('smiles'):
                pred_df.loc[sdf.index, 'snum'] = len(sdf)
            pred_df = pred_df[pred_df['snum'] >solvent]
            
        pred_df['err'] = abs(pred_df['pred'] - pred_df[target])
        mae = pred_df['err'].mean()
        mse = ((pred_df['err']) ** 2).mean()
        rmse =  mse ** .5
        slope, intercept, r_value, p_value, std_err = stats.linregress(pred_df['pred'], pred_df[target])
        r2 = r_value**2
#         r2 = 1-((pred_df['err']) ** 2).sum()/((pred_df[target]-pred_df[target].mean())**2).sum()
        res.append([round(mae,3), round(mse,3), round(rmse,3), round(r2,3), len(pred_df),len(pred_df['smiles'].unique())])
res = np.array(res)

In [ ]:
pd.DataFrame(res,index = pd.MultiIndex.from_tuples(idx),
               columns =['MAE', 'MSE', 'RMSE', 'R2', 'n_data','n_mol']
         )

In [ ]:
# pred_df = pred_df[pred_df[target] >3.5]
# pred_df = pred_df[pred_df[target] <6]

pd.DataFrame(res,index = pd.MultiIndex.from_tuples(idx),
               columns =['MAE', 'MSE', 'RMSE', 'R2', 'n_data','n_mol']
         )